# Preparación de datos · Núcleo del proyecto de vivienda

**Objetivo:** transformar cuatro fuentes oficiales (IPV, IPC, salarios y valor tasado) en tablas limpias, homogéneas y listas para cargar en SQL.

**Reglas de trabajo:** los archivos de `data/raw/core/` permanecen intactos; se conserva la granularidad de cada fuente; los valores ausentes no se imputan; los CSV finales se guardan en `data/processed/core/`. Ejecutar las celdas **en orden**, desde el inicio, con el notebook ubicado en `notebooks/`.

| Fuente | Archivo original | Resultado | Frecuencia |
|---|---|---|---|
| IPV (INE, base 2025) | `base25_raw.xlsx` | `ipv.csv` | Trimestral |
| IPC (INE, base 2025) | `ipcgrupos_raw.xlsx` | `ipc.csv` | Mensual |
| Ganancia media anual (INE) | `salarios_raw.xlsx` | `salarios.csv` | Anual |
| Valor tasado, vivienda libre | `tasa_raw.csv` | `tasa.csv` | Trimestral |

> **Interpretación:** el IPV mide la evolución relativa de precios de compraventa; el IPC mide precios de consumo; el salario es una ganancia anual **bruta media por trabajador**; el valor tasado se expresa en **€/m²**. Son indicadores distintos: no se deben interpretar como si midieran lo mismo.

## 0. Configuración y utilidades

Se definen las rutas y un catálogo común de territorios a partir del archivo IPV. El código autonómico se conserva como **texto** (`"01"`…`"19"`); el agregado `Nacional` no tiene código. No se alteran los archivos originales.

In [ ]:
from pathlib import Path
import pandas as pd

# Este notebook se ejecuta desde la carpeta notebooks/ del repositorio.
RAW = Path("../data/raw/core")
PROCESSED = Path("../data/processed/core")
PROCESSED.mkdir(parents=True, exist_ok=True)


def guardar_y_verificar(df, nombre, nulos_esperados=None):
    """Exporta a CSV y verifica filas, columnas y nulos al leerlo de nuevo."""
    destino = PROCESSED / nombre
    df.to_csv(destino, index=False, encoding="utf-8-sig", na_rep="")
    comprobacion = pd.read_csv(destino, dtype={"cod_territorio": "string"})
    assert comprobacion.shape == df.shape, f"Dimensiones distintas tras exportar {nombre}"
    assert comprobacion.columns.tolist() == df.columns.tolist()
    if nulos_esperados is not None:
        columna, cantidad = nulos_esperados
        assert comprobacion[columna].isna().sum() == cantidad
    print(f"Exportado: {destino} · {len(comprobacion):,} filas")
    display(comprobacion.head(3))

---
## 1. Índice de Precios de Vivienda (IPV)

**Fuente:** INE, serie trimestral con base 2025. Se utiliza únicamente el bloque **índice**; los bloques de variación trimestral, anual y acumulada no son necesarios para la tabla de trabajo.

**Estructura original:** fila 7 → periodos; filas 8–87 → 20 bloques territoriales de 4 filas (etiqueta territorial, general, nueva y segunda mano). En el primer bloque de columnas, la columna 0 contiene etiquetas y las columnas 1–78 los índices.

### 1.1 Lectura y selección

`header=None` conserva la disposición original del Excel. La selección excluye notas y otros bloques de variación.

In [ ]:
df_ipv = pd.read_excel(RAW / "base25_raw.xlsx", header=None)

periodos_ipv = df_ipv.iloc[7, 1:79].tolist()  # 78 trimestres
ipv_indices = df_ipv.iloc[8:88, :79].copy()

# Cada cuatro filas, la primera indica el territorio.
ipv_indices["territorio"] = ipv_indices.iloc[:, 0].where(
    (ipv_indices.index - 8) % 4 == 0
).ffill()

# Conservar las otras tres filas (general, nueva y segunda mano).
ipv_indices = ipv_indices.loc[(ipv_indices.index - 8) % 4 != 0].copy()
ipv_indices.columns = ["tipo_vivienda", *periodos_ipv, "territorio"]

assert ipv_indices.shape == (60, 80)
display(ipv_indices.iloc[:6, :6])

### 1.2 Normalización y formato largo

Una observación pasa a ser un territorio, un trimestre y un tipo de vivienda. El catálogo territorial se construye **desde las etiquetas sin modificar del Excel**: así la extracción de códigos no depende de ejecutar una celda más de una vez.

In [ ]:
# Catálogo territorial original, reutilizado en las otras tres fuentes.
catalogo_territorios = pd.DataFrame({
    "territorio_original": ipv_indices["territorio"].drop_duplicates().tolist()
})
catalogo_territorios["cod_territorio"] = (
    catalogo_territorios["territorio_original"].str.extract(r"^(\d{2})", expand=False)
)
catalogo_territorios["territorio"] = (
    catalogo_territorios["territorio_original"]
    .str.replace(r"^\d{2}\s+", "", regex=True)
)
mapa_territorios = catalogo_territorios.set_index("territorio")["cod_territorio"]

ipv_largo = ipv_indices.melt(
    id_vars=["territorio", "tipo_vivienda"],
    var_name="periodo", value_name="indice_ipv"
)
ipv_largo["territorio"] = ipv_largo["territorio"].str.replace(
    r"^\d{2}\s+", "", regex=True
)
ipv_largo["cod_territorio"] = ipv_largo["territorio"].map(mapa_territorios)
ipv_largo["tipo_vivienda"] = ipv_largo["tipo_vivienda"].str.strip()
ipv_largo["anio"] = ipv_largo["periodo"].str[:4].astype(int)
ipv_largo["trimestre"] = ipv_largo["periodo"].str[-1].astype(int)
ipv_largo["indice_ipv"] = pd.to_numeric(ipv_largo["indice_ipv"], errors="coerce")

ipv_largo = ipv_largo[[
    "cod_territorio", "territorio", "anio", "trimestre", "tipo_vivienda", "indice_ipv"
]].sort_values(
    ["cod_territorio", "anio", "trimestre", "tipo_vivienda"], na_position="first"
).reset_index(drop=True)

display(ipv_largo.head(3))

### 1.3 Calidad y ausencias

**Resultado verificado durante la exploración:** 4.680 registros (20 territorios × 3 tipos × 78 trimestres). Hay **312 índices ausentes**, todos en las series *vivienda nueva* y *vivienda de segunda mano* de **Ceuta y Melilla**. Se mantienen como nulos; no se interpolan ni eliminan.

In [ ]:
clave_ipv = ["territorio", "anio", "trimestre", "tipo_vivienda"]
nulos_ipv = ipv_largo.loc[ipv_largo["indice_ipv"].isna()]

assert len(ipv_largo) == 4680
assert ipv_largo["territorio"].nunique() == 20
assert ipv_largo["tipo_vivienda"].nunique() == 3
assert ipv_largo.duplicated(clave_ipv).sum() == 0
assert len(nulos_ipv) == 312
assert set(nulos_ipv["territorio"]) == {"Ceuta", "Melilla"}
assert set(nulos_ipv["tipo_vivienda"]) == {"Vivienda nueva", "Vivienda segunda mano"}
assert ipv_largo.loc[ipv_largo["territorio"] != "Nacional", "cod_territorio"].notna().all()
assert ipv_largo.loc[ipv_largo["territorio"] == "Nacional", "cod_territorio"].isna().all()
print("IPV: validaciones superadas")

### 1.4 Exportación

In [ ]:
guardar_y_verificar(ipv_largo, "ipv.csv", ("indice_ipv", 312))

---
## 2. Índice de Precios de Consumo (IPC)

**Fuente:** INE, índices mensuales, base 2025. Se conserva el bloque de **índices**: columnas 1–297, correspondientes a 297 meses. Cada territorio ocupa **15 filas**: nombre territorial, índice general y 13 grupos de consumo. La columna de gasto **«04 Vivienda, agua, electricidad…» no es un índice de precios de compraventa de vivienda**.

### 2.1 Lectura y extracción de los indicadores

In [ ]:
df_ipc = pd.read_excel(RAW / "ipcgrupos_raw.xlsx", header=None)

periodos_ipc = df_ipc.iloc[7, 1:298].tolist()
ipc_indices = df_ipc.iloc[8:308, :298].copy()

# Primera fila de cada bloque de 15: nombre territorial.
ipc_indices["territorio"] = ipc_indices.iloc[:, 0].where(
    (ipc_indices.index - 8) % 15 == 0
).ffill()
ipc_indices = ipc_indices.loc[(ipc_indices.index - 8) % 15 != 0].copy()
ipc_indices.columns = ["grupo_consumo", *periodos_ipc, "territorio"]

assert ipc_indices.shape == (280, 299)
display(ipc_indices.iloc[:4, :5])

### 2.2 Normalización y formato largo

Se separan año y mes y se reutiliza el mapa de territorios construido desde IPV. Esto evita extraer códigos sobre nombres que ya se han normalizado.

In [ ]:
ipc_largo = ipc_indices.melt(
    id_vars=["territorio", "grupo_consumo"],
    var_name="periodo", value_name="indice_ipc"
)
ipc_largo["territorio"] = ipc_largo["territorio"].str.replace(
    r"^\d{2}\s+", "", regex=True
)
ipc_largo["cod_territorio"] = ipc_largo["territorio"].map(mapa_territorios)
ipc_largo["grupo_consumo"] = ipc_largo["grupo_consumo"].str.strip()
ipc_largo["anio"] = ipc_largo["periodo"].str[:4].astype(int)
ipc_largo["mes"] = ipc_largo["periodo"].str[-2:].astype(int)
ipc_largo["indice_ipc"] = pd.to_numeric(ipc_largo["indice_ipc"], errors="coerce")

ipc_largo = ipc_largo[[
    "cod_territorio", "territorio", "anio", "mes", "grupo_consumo", "indice_ipc"
]].sort_values(
    ["cod_territorio", "anio", "mes", "grupo_consumo"], na_position="first"
).reset_index(drop=True)

display(ipc_largo.head(3))

### 2.3 Calidad y exportación

**Resultado verificado:** 83.160 registros (20 territorios × 14 indicadores × 297 meses), sin índices nulos y sin duplicados por territorio, año, mes y grupo.

In [ ]:
clave_ipc = ["territorio", "anio", "mes", "grupo_consumo"]

assert len(ipc_largo) == 83160
assert ipc_largo["territorio"].nunique() == 20
assert ipc_largo["grupo_consumo"].nunique() == 14
assert ipc_largo.duplicated(clave_ipc).sum() == 0
assert ipc_largo["indice_ipc"].notna().all()
assert ipc_largo["mes"].between(1, 12).all()
assert ipc_largo.loc[ipc_largo["territorio"] != "Nacional", "cod_territorio"].notna().all()
assert ipc_largo.loc[ipc_largo["territorio"] == "Nacional", "cod_territorio"].isna().all()
print("IPC: validaciones superadas")

guardar_y_verificar(ipc_largo, "ipc.csv", ("indice_ipc", 0))

---
## 3. Ganancia media anual por trabajador (salarios)

**Fuente:** INE, **ganancia media anual bruta por trabajador**, ambos sexos, todos los sectores. La tabla incluye el agregado nacional y 17 comunidades autónomas; **no contiene Ceuta ni Melilla**. Abarca **2008–2024**, por lo que las comparaciones con el IPV deben realizarse sobre periodos comunes.

En el Excel, la fila 6 contiene los nombres territoriales y, desde la fila 8, se alternan filas de año y filas con valores de «Ambos sexos».

### 3.1 Lectura y selección

In [ ]:
df_salarios = pd.read_excel(RAW / "salarios_raw.xlsx", header=None)

territorios_salarios = df_salarios.iloc[6, 1:].tolist()
anios_salarios = df_salarios.iloc[8:42:2, 0].astype(str).str.strip().astype(int).tolist()
valores_salarios = df_salarios.iloc[9:43:2, 1:].copy()

assert len(territorios_salarios) == 18
assert len(anios_salarios) == 17
assert valores_salarios.shape == (17, 18)

# Las filas pares del bloque son años y las impares, salarios.
salarios_ancho = valores_salarios.copy()
salarios_ancho.columns = territorios_salarios
salarios_ancho["anio"] = anios_salarios

### 3.2 Formato largo y normalización

Se sustituye `Total Nacional` por `Nacional`, se convierte el salario a numérico y se asignan los códigos territoriales existentes. No se crea ningún registro para Ceuta o Melilla.

In [ ]:
salarios_largo = salarios_ancho.melt(
    id_vars="anio", var_name="territorio", value_name="salario_medio"
)
salarios_largo["territorio"] = (
    salarios_largo["territorio"].str.strip().replace({"Total Nacional": "Nacional"})
)
salarios_largo["salario_medio"] = pd.to_numeric(
    salarios_largo["salario_medio"], errors="coerce"
)
salarios_largo["cod_territorio"] = salarios_largo["territorio"].map(mapa_territorios)

salarios_largo = salarios_largo[[
    "cod_territorio", "territorio", "anio", "salario_medio"
]].sort_values(["cod_territorio", "anio"], na_position="first").reset_index(drop=True)

display(salarios_largo.head(3))

### 3.3 Calidad y exportación

**Resultado verificado:** 306 observaciones (18 territorios × 17 años), sin salarios nulos, duplicados ni valores no positivos. Un valor de salario medio anual **no equivale al ingreso familiar ni al salario neto**.

In [ ]:
assert len(salarios_largo) == 306
assert salarios_largo["territorio"].nunique() == 18
assert salarios_largo["anio"].nunique() == 17
assert salarios_largo["anio"].between(2008, 2024).all()
assert salarios_largo["salario_medio"].notna().all()
assert (salarios_largo["salario_medio"] > 0).all()
assert salarios_largo.duplicated(["territorio", "anio"]).sum() == 0
assert salarios_largo.loc[salarios_largo["territorio"] != "Nacional", "cod_territorio"].notna().all()
assert salarios_largo.loc[salarios_largo["territorio"] == "Nacional", "cod_territorio"].isna().all()
print("Salarios: validaciones superadas")

guardar_y_verificar(salarios_largo, "salarios.csv", ("salario_medio", 0))

---
## 4. Valor tasado de la vivienda libre (€/m²)

**Fuente:** serie trimestral 1995–2026. El archivo CSV contiene los regímenes `Libre` y `Protegida`, además de **niveles provincial, autonómico y nacional**. Para este análisis se usa **vivienda libre**, en nivel autonómico y nacional.

**Incidencias de la fuente detectadas en la exploración:**
- Algunas comunidades uniprovinciales y las ciudades autónomas aparecen con el nombre provincial, no con `Provincia = Total CCAA`.
- Navarra presenta **110 registros exactamente duplicados** (sin conflictos entre valores en un mismo trimestre).
- Ceuta y Melilla tienen **36 nulos originales cada una** (1995–2003).
- Melilla, **2013T3**, figura con **0 €/m²**; se reclasifica como dato ausente por no ser un valor tasado válido. No se presupone la causa de origen.

### 4.1 Lectura y filtrado del nivel territorial

Se combina `Provincia = Total CCAA` con cinco excepciones identificadas en el archivo original. No se agregan precios provinciales mediante medias.

In [ ]:
df_tasa = pd.read_csv(RAW / "tasa_raw.csv", sep=";", encoding="utf-8-sig")

# En estas cinco áreas, el registro disponible está etiquetado como provincia.
excepciones_uniprovinciales = [
    "Balears, Illes", "Cantabria", "Rioja, La", "Ceuta", "Melilla"
]
tasa = df_tasa.loc[
    df_tasa["Régimen"].eq("Libre")
    & (
        df_tasa["Provincia"].eq("Total CCAA")
        | df_tasa["Comunidad_Autónoma"].isin(excepciones_uniprovinciales)
    )
].copy()

assert tasa["Comunidad_Autónoma"].nunique() == 20
assert tasa.duplicated().sum() == 110

# Antes de descartar duplicados, comprobar que no hay valores contradictorios.
conflictos = tasa.groupby(
    ["Comunidad_Autónoma", "Año", "Trimestre"]
)["Valor"].nunique(dropna=False)
assert (conflictos <= 1).all(), "Hay valores contradictorios para algún trimestre"

# Únicamente copias exactas: no se pierden observaciones distintas.
tasa = tasa.drop_duplicates().copy()
assert len(tasa) == 2520
print("Tasaciones: 110 duplicados exactos eliminados")

### 4.2 Normalización territorial y columnas definitivas

Se armonizan tres denominaciones que difieren de las del INE y se asignan los códigos territoriales usando el catálogo común.

In [ ]:
tasa["territorio"] = tasa["Comunidad_Autónoma"].replace({
    "Total nacional": "Nacional",
    "Castilla-La Mancha": "Castilla - La Mancha",
    "Murcia, Region de": "Murcia, Región de"
})
tasa["cod_territorio"] = tasa["territorio"].map(mapa_territorios)

tasa_limpia = tasa[[
    "cod_territorio", "territorio", "Año", "Trimestre", "Valor"
]].rename(columns={
    "Año": "anio", "Trimestre": "trimestre", "Valor": "valor_m2"
}).copy()

# Asegurar columnas numéricas y orden cronológico.
tasa_limpia["valor_m2"] = pd.to_numeric(tasa_limpia["valor_m2"], errors="coerce")
tasa_limpia = tasa_limpia.sort_values(
    ["cod_territorio", "anio", "trimestre"], na_position="first"
).reset_index(drop=True)

### 4.3 Tratamiento explícito de valores ausentes

Los **72 nulos originales** se conservan. El valor cero de Melilla en 2013T3 se convierte en nulo **antes** de validar y exportar; en el CSV final habrá **73 ausencias**. Este tratamiento debe constar en el informe de calidad de datos.

In [ ]:
# Identificar la única observación con cero antes de reclasificarla.
ceros = tasa_limpia.loc[tasa_limpia["valor_m2"].eq(0)]
assert len(ceros) == 1
assert (
    (ceros["territorio"] == "Melilla")
    & (ceros["anio"] == 2013)
    & (ceros["trimestre"] == 3)
).all()
assert tasa_limpia["valor_m2"].isna().sum() == 72

# Precio tasado de 0 €/m²: señal de ausencia, no precio interpretable.
tasa_limpia.loc[tasa_limpia["valor_m2"].eq(0), "valor_m2"] = float("nan")

print("Nulos finales por territorio:")
print(tasa_limpia.loc[tasa_limpia["valor_m2"].isna()].groupby("territorio").size())

### 4.4 Calidad y exportación

**Resultado verificado:** 2.520 observaciones (20 territorios × 126 trimestres), 73 valores ausentes y ninguna clave territorial-temporal duplicada. El fichero se exporta **después** del tratamiento del cero de Melilla.

In [ ]:
assert len(tasa_limpia) == 2520
assert tasa_limpia["territorio"].nunique() == 20
assert tasa_limpia["valor_m2"].isna().sum() == 73
assert (tasa_limpia["valor_m2"].dropna() > 0).all()
assert tasa_limpia["trimestre"].between(1, 4).all()
assert tasa_limpia.duplicated(["territorio", "anio", "trimestre"]).sum() == 0
assert tasa_limpia.loc[tasa_limpia["territorio"] != "Nacional", "cod_territorio"].notna().all()
assert tasa_limpia.loc[tasa_limpia["territorio"] == "Nacional", "cod_territorio"].isna().all()
assert set(tasa_limpia.loc[tasa_limpia["valor_m2"].isna(), "territorio"]) == {"Ceuta", "Melilla"}
print("Tasaciones: validaciones superadas")

guardar_y_verificar(tasa_limpia, "tasa.csv", ("valor_m2", 73))

---
## 5. Resumen y siguientes pasos

| Tabla procesada | Filas esperadas | Valores nulos de la medida | Cobertura observada |
|---|---:|---:|---|
| `ipv.csv` | 4.680 | 312 | 2007T1–2026T2 |
| `ipc.csv` | 83.160 | 0 | 2002M01–2026M09 |
| `salarios.csv` | 306 | 0 | 2008–2024 |
| `tasa.csv` | 2.520 | 73 | 1995T1–2026T2 |

**Decisiones que deben figurar en `docs/informe_exploracion.md` o en un informe de limpieza:** diferencias de granularidad y cobertura temporal; ausencia de Ceuta/Melilla en salarios; 312 nulos de IPV; 110 duplicados exactos de Navarra eliminados; 72 nulos de tasación históricos; y conversión del cero de Melilla 2013T3 a nulo.

**Para SQL:** cargar los códigos territoriales como texto y conservar las celdas de medida vacías como `NULL`; no sumar índices de precios ni comparar sin armonizar frecuencia y horizonte temporal. Antes de la carga definitiva, contrastar una muestra de registros con cada fuente original.